# Climate Intelligence Engine

Pipeline analítico para detecção de inconsistências, completamento de dados e otimização de precisão climática.

Execute cada célula em sequência para rodar o pipeline completo de pré-processamento.

In [ ]:
import subprocess
import sys

def run(script):
    print(f"▶  {script}\n{'─' * 50}")
    result = subprocess.run([sys.executable, script], capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"{script} falhou (código {result.returncode})")
    print("✓  Concluído\n")

## 1.1 — Download dos Dados

Baixa `stations.parquet` e `weather_measurements.parquet` do Google Drive para `data/`.

In [ ]:
run("1.1_download_data.py")

## 1.2 — Cálculo de Distâncias entre Estações

Gera `data/station_distances.parquet` com distâncias par-a-par (Haversine + altitude) entre as 663 estações.

> Requer: 1.1

In [ ]:
run("1.2_compute_station_distances.py")

## 1.3 — Análise Exploratória

Executa o notebook de EDA e exporta o relatório PDF para `doc/1.3_exploratory_analysis.pdf`.

> Requer: 1.1

In [ ]:
# print("▶  1.3_exploratory_analysis.ipynb\n" + "─" * 50)
# result = subprocess.run(
#     ["jupyter", "nbconvert", "--to", "notebook", "--execute",
#      "--inplace", "--ExecutePreprocessor.timeout=600",
#      "1.3_exploratory_analysis.ipynb"],
#     capture_output=True, text=True,
# )
# print(result.stdout or result.stderr)
# if result.returncode != 0:
#     raise RuntimeError(f"1.3 falhou (código {result.returncode})")
# print("✓  Concluído\n")

## 1.4 — Limpeza e Separação das Variáveis

Gera um Parquet por variável (`temperature`, `humidity`, `rainfall`, `global_radiation`, `pressure`) com schema `code | time | measurement`.

> Requer: 1.1

In [ ]:
run("1.4_clean_data.py")

## 1.5 — Enriquecimento com Vizinhos

Para cada variável, enriquece cada medição com os 20 vizinhos mais próximos disponíveis no mesmo timestamp.

Schema de saída (107 colunas): `code | time | measurement | n01..n20 | d01..d20 | a01..a20 | b01_sin/cos..b20_sin/cos | hour_sin/cos | doy_sin/cos`

> Requer: 1.1, 1.2, 1.4

In [ ]:
run("1.5_build_neighbors.py")

## 1.6 — Normalização das Features (StandardScaler)

Aplica StandardScaler (µ=0, σ=1) em todas as colunas numéricas dos arquivos `_neighbors.parquet`. Salva o parquet escalado em `data/` e o scaler em `models/1.6_scaler_{variable}.json`.

> Requer: 1.5

In [ ]:
run("1.6_scale_features.py")

## 2.0 — Neighbors (Baseline: Vizinho Mais Próximo)

Avalia o vizinho mais próximo (n01) como estimador da medição real no conjunto de teste. Gera `results/2.0_neighbors/metrics.csv` com MAE, RMSE, R², Bias e r por variável.

> Requer: 1.6

In [ ]:
run("2.0_neighbors.py")

## 3.0 — Regressão Linear

Treina OLS no conjunto de treino e avalia no teste. Salva `results/3.0_linear_regression/{variable}/model.npy` e `metrics.csv` por variável.

> Requer: 1.6

In [ ]:
run("3.0_linear_regression.py")

## 4.0 — Rede Neural Densa (MLP)

MLP com arquitetura `79 → 256 → 512 → 256 → 128 → 64 → 1` (BatchNorm + GELU + Dropout).
AdamW + Huber loss + ReduceLROnPlateau. Early stopping com patience=25 épocas.
Detecta GPUs automaticamente — processa uma variável por GPU em paralelo.

Salva `models/{variable}_dense.pt` (melhor val MAE) e `results/4.0_dense_layer/`.

> Requer: 1.6

In [ ]:
run("4.0_dense_layer.py")